In [ ]:
!python --version

Python 3.12.12


In [ ]:
! git clone https://github.com/Sauravroy34/DeepRetro.git

Cloning into 'DeepRetro'...
remote: Enumerating objects: 12507, done.
remote: Counting objects: 100% (2567/2567), done.
remote: Compressing objects: 100% (741/741), done.
remote: Total 12507 (delta 2141), reused 1829 (delta 1826), pack-reused 9940 (from 2)
Receiving objects: 100% (12507/12507), 20.72 MiB | 14.55 MiB/s, done.
Resolving deltas: 100% (5756/5756), done.


In [ ]:
%cd DeepRetro

/content/DeepRetro


In [ ]:
! python -m pip install aizynthfinder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.8/34.8 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.3 kB 15.7 MB/s eta 0:00:00
   ━

In [ ]:
!mkdir -p aizynthfinder/models/USPTO
!python -m aizynthfinder.tools.download_public_data aizynthfinder/models/USPTO/

uspto_model.onnx: 100% 91.5M/91.5M [01:15<00:00, 1.22MB/s]
uspto_templates.csv.gz: 3.31MB [00:04, 723kB/s]
uspto_ringbreaker_model.onnx: 100% 15.0M/15.0M [00:24<00:00, 610kB/s]
uspto_ringbreaker_templates.csv.gz: 374kB [00:00, 461kB/s]
zinc_stock.hdf5: 100% 663M/663M [00:33<00:00, 19.9MB/s]
uspto_filter_model.onnx: 100% 16.8M/16.8M [00:16<00:00, 1.02MB/s]
Configuration file written to config.yml


In [ ]:
%%writefile .env
AZ_MODELS_PATH= aizynthfinder/models
AZ_MODEL_CONFIG_PATH=aizynthfinder/models/USPTO/config.yml


Writing .env


In [ ]:
! ls

aizynthfinder	    logs		 retro.code-workspace
config		    model_out		 scripts
data		    molecules.txt	 src
deepretro	    name.txt		 start_backend.sh
docker-compose.yml  notebooks		 test_hallucination_model_trainer.ipynb
Dockerfile	    pyproject.toml	 test_langfuse_config.py
docs		    reaction_prediction  tests
env.example	    readme.md		 uv.lock
environment.yml     results		 viewer


In [ ]:
! pip install structlog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.3 MB/s eta 0:00:00


In [ ]:
!pip install -U rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 24.0 MB/s eta 0:00:00
  Attempting uninstall: rdkit
    Found existing installation: rdkit 2023.9.6
    Uninstalling rdkit-2023.9.6:
      Successfully uninstalled rdkit-2023.9.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
reaction-utils 1.9.4 requires rdkit<2024.9.1,>=2023.9.1; python_version < "3.13", but you have rdkit 2026.3.6 which is incompatible.
aizynthfinder 4.4.1 requires rdkit<2024.0.0,>=2023.9.1, but you have rdkit 2026.3.6 which is incompatible.


# NOTE MAKE SURE TO RESTART SESSION OVER HERE

# MultiStep Retrosynthesis


## Global vars over here

In [ ]:
%%writefile deepretro/utils/hf.py
import torch
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM



def build_chat_prompt(tokenizer, user_prompt, SYS_PROMPT, molecule_smiles=None):

    messages = [
        {
            "role": "system",
            "content": SYS_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt.replace('{target_smiles}', molecule_smiles),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def generate(model_name, smiles, user_prompt, top_p, max_tokens, SYS_PROMPT, temperature):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    ).to(device)
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # left side padding for generation https://discuss.huggingface.co/t/the-effect-of-padding-side/67188
    model.config.pad_token_id = tokenizer.pad_token_id

    prompt = build_chat_prompt(tokenizer, user_prompt, SYS_PROMPT, smiles)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=1,
            max_new_tokens=max_tokens,
            do_sample=False,
            top_p=top_p,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]

    out = outputs[:, prompt_len:]
    out = tokenizer.decode(out[0], skip_special_tokens=True)
    return out

Overwriting deepretro/utils/hf.py


In [ ]:
%%writefile src/utils/llm.py

import os
import ast
import litellm
from typing import Optional
from dotenv import load_dotenv
from litellm import completion
from src.variables import OPENAI_MODELS, DEEPSEEK_MODELS
from src.variables import USER_PROMPT, SYS_PROMPT
from src.variables import USER_PROMPT_V4, SYS_PROMPT_V4
from src.variables import USER_PROMPT_OPENAI, SYS_PROMPT_OPENAI
from src.variables import USER_PROMPT_DEEPSEEK, SYS_PROMPT_DEEPSEEK
from src.variables import ADDON_PROMPT_7_MEMBER, USER_PROMPT_DEEPSEEK_V4
from src.variables import ERROR_MAP, PROTECTING_GROUP_CONTEXT
from src.variables import CLAUDE_EXTENDED_THINKING_MODELS
from src.variables import EXTENDED_THINKING_BUDGET_TOKENS
from src.variables import EXTENDED_THINKING_MAX_TOKENS
from src.cache import cache_results
from src.utils.utils_molecule import validity_check, detect_seven_member_rings
from src.utils.job_context import logger as context_logger
from src.utils.stability_checks import stability_checker
from src.utils.hallucination_checks import hallucination_checker
from src.protecting_group import mask_protecting_groups_multisymbol

load_dotenv()

# set the success callback to langfuse for logging
litellm.success_callback = ["langfuse"]
litellm.drop_params = True

from src.utils.langfuse_config import get_langfuse_metadata

ENABLE_LOGGING = False if os.getenv("ENABLE_LOGGING",
                                    "true").lower() == "false" else True


def log_message(message: str, logger=None):
    """Log the message

    Parameters
    ----------
    message : str
        The message to be logged
    logger : _type_, optional
        The logger object, by default None

    Returns
    -------
    None
    """
    if logger is not None:
        logger.info(message)
    else:
        print(message)


def supports_extended_thinking(LLM: str) -> bool:
    """Check whether the model supports Anthropic extended thinking.

    Parameters
    ----------
    LLM : str
        The LLM model identifier, possibly with a provider prefix
        (e.g. "anthropic/claude-sonnet-4-20250514") and/or an ":adv"
        advanced-prompt suffix.

    Returns
    -------
    bool
        True if extended thinking should be enabled for this model
    """
    model_name = LLM.split("/")[-1].split(":")[0]
    return any(known in model_name
               for known in CLAUDE_EXTENDED_THINKING_MODELS)


def obtain_prompt(LLM: str):
    """Obtain the prompt based on the LLM model

    Parameters
    ----------
    LLM : str
        The LLM model to be used

    Returns
    -------
    str, str, int
        The system prompt, user prompt and max completion tokens
    """
    advanced_prompt = False
    detector = LLM.split(":")
    if len(detector) > 1 and detector[1] == "adv":
        advanced_prompt = True
    print(f"Advanced Prompt: {advanced_prompt}")
    if advanced_prompt:
        if LLM in DEEPSEEK_MODELS:
            sys_prompt_final = SYS_PROMPT_V4
            user_prompt_final = USER_PROMPT_DEEPSEEK_V4
            max_completion_tokens = 8192 * 2
        elif LLM in OPENAI_MODELS:
            sys_prompt_final = SYS_PROMPT_OPENAI
            user_prompt_final = USER_PROMPT_OPENAI
            max_completion_tokens = 8192
        else:
            sys_prompt_final = SYS_PROMPT_V4
            user_prompt_final = USER_PROMPT_V4
            max_completion_tokens = 4096
    else:
        if LLM in DEEPSEEK_MODELS:
            sys_prompt_final = SYS_PROMPT_DEEPSEEK
            user_prompt_final = USER_PROMPT_DEEPSEEK
            max_completion_tokens = 8192 * 2
        elif LLM in OPENAI_MODELS:
            sys_prompt_final = SYS_PROMPT_OPENAI
            user_prompt_final = USER_PROMPT_OPENAI
            max_completion_tokens = 8192
        else:
            sys_prompt_final = SYS_PROMPT
            user_prompt_final = USER_PROMPT
            max_completion_tokens = 4096
    return sys_prompt_final, user_prompt_final, max_completion_tokens


@cache_results
def call_LLM(molecule: str,
             LLM: str = "claude-opus-4-8",
             temperature: float = 0.0,
             messages: Optional[list[dict]] = None,
             use_protecting_group_feature: bool = False,
             local: bool = False) -> tuple[int, str]:
    """Calls the LLM model to predict the next step

    Parameters
    ----------
    molecule : str
        The target molecule for retrosynthesis
    LLM : str, optional
        The LLM model to be used, by default "claude-opus-4-8"
    temperature : float, optional
        The temperature for sampling, by default 0.0
    messages : Optional[list[dict]], optional
        The conversation history, by default None

    Returns
    -------
    tuple[int, str]
        The status code and the response text
    """
    logger = context_logger.get() if ENABLE_LOGGING else None
    log_message(f"Calling {LLM} with molecule: {molecule}", logger)

    # Check for seven-member rings
    if detect_seven_member_rings(molecule):
        log_message(f"Detected seven member ring in molecule: {molecule}",
                    logger)
        add_on = ADDON_PROMPT_7_MEMBER
    else:
        add_on = ""

    # Check for protecting groups and add context
    if use_protecting_group_feature:
        masked_smiles = mask_protecting_groups_multisymbol(molecule)
        if masked_smiles != molecule and masked_smiles != "INVALID_SMILES":
            log_message(
                f"Detected protecting groups in molecule: {molecule} -> {masked_smiles}",
                logger)
            protecting_group_context = PROTECTING_GROUP_CONTEXT.format(
                molecule=molecule, masked_smiles=masked_smiles)
            add_on += protecting_group_context

    sys_prompt_final, user_prompt_final, max_completion_tokens = obtain_prompt(
        LLM)
    LLM = LLM.split(":")[0]

    params = {
        "model": LLM,
        "max_completion_tokens": max_completion_tokens,
        "temperature": temperature,
        "seed": 42,
        "top_p": 0.9,
        "metadata": get_langfuse_metadata("retrosynthesis"),
        "drop_params": True,
    }

    if LLM in DEEPSEEK_MODELS:
        user_prompt_final += add_on

    if supports_extended_thinking(LLM):
        params["max_tokens"] = EXTENDED_THINKING_MAX_TOKENS
        params["temperature"] = 1
        params.pop("top_p", None)
        params.pop("max_completion_tokens", None)
        try:
            is_adaptive_thinking_model = "claude-opus-4-8" in LLM or bool(
                litellm.get_model_info(LLM).get("supports_adaptive_thinking"))
        except Exception:
            is_adaptive_thinking_model = False
        if is_adaptive_thinking_model:
            params['thinking'] = {"type": "adaptive"}
        else:
            params['thinking'] = {
                "type": "enabled",
                "budget_tokens": EXTENDED_THINKING_BUDGET_TOKENS
            }

    params = {k: v for k, v in params.items() if LLM != "claude-opus-4-8" or k not in ("temperature", "top_p", "seed")}
    if messages is None:
        messages = [{
            "role": "system",
            "content": sys_prompt_final + add_on
        }, {
            "role":
            "user",
            "content":
            user_prompt_final.replace('{target_smiles}', molecule) + "\n\n" +
            add_on
        }]
    params["messages"] = messages


    if local:
        params["user_prompt"] = user_prompt_final
        params["sys_prompt"] = sys_prompt_final



    if local:
        from deepretro.utils.hf import generate

        response = generate(
            model_name=params["model"],
            smiles=molecule,
            user_prompt=user_prompt_final,
            top_p=params["top_p"],
            max_tokens=params["max_completion_tokens"],
            SYS_PROMPT=sys_prompt_final,
            temperature=params["temperature"],
        )

        return 200, response

    try:
        # Call the LLM model
        response = completion(**params)

        res_text = response.choices[0].message.content
    except Exception as e:
        log_message(f"Error in calling {LLM}: {e}", logger)
        log_message(f"Retrying call to {LLM}", logger)
        try:
            response = completion(**params)
            res_text = response.choices[0].message.content
        except Exception as e:
            log_message(f"2nd Error in calling {LLM}: {e}", logger)
            log_message(f"Exiting call to {LLM}", logger)
            return 400, ""
    log_message(f"Received response from LLM: {res_text}", logger)
    return 200, res_text



def strip_thinking(res_text: str) -> str:
    """Drop the reasoning block so the parser only sees the answer."""
    _THINK_END = "</think>"
    i = res_text.rfind(_THINK_END)
    return res_text[i + len(_THINK_END) :] if i != -1 else res_text


def split_json_content(res_text: str):


    """Return (status, json_content). 200 on success, 502 on failure."""
    res_text = strip_thinking(res_text)

    # 1. Match explicit <json>...</json> tags
    start = res_text.find("<json>")
    end = res_text.find("</json>")
    if start != -1 and end != -1 and end > start:
        content = res_text[start + len("<json>") : end].strip()
        if content:
            return 200, content

    # 2. Match standard ```json ... ``` markdown blocks
    if "```json" in res_text:
        s = res_text.find("```json") + len("```json")
        e = res_text.find("```", s)
        if e != -1:
            content = res_text[s:e].strip()
            if content:
                return 200, content

    # 3. Non-greedy JSON salvage: find outer braces { ... }
    first_brace = res_text.find("{")
    last_brace = res_text.rfind("}")
    if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
        content = res_text[first_brace : last_brace + 1].strip()
        if content:
            return 200, content

    return 502, ""

def split_cot_json(res_text: str) -> tuple[int, list[str], str]:
    """Parse the LLM response to extract the thinking steps and json content

    Parameters
    ----------
    res_text : str
        The response text from the LLM model

    Returns
    -------
    tuple[int, list[str], str]
        The status code, thinking steps and json content
    """
    logger = context_logger.get() if ENABLE_LOGGING else None
    try:
        # extract the content within <cot> </cot> tags as thinking content
        thinking_content = res_text[res_text.find("<cot>\n") +
                                    6:res_text.find("</cot>")]
        if not thinking_content:
            return 501, [], ""

        # split the thinking content into individual steps based on the <thinking> </thinking> tags
        thinking_steps = thinking_content.split("<thinking")[1:]
        thinking_steps = [
            step[:step.find("</thinking>")] for step in thinking_steps
        ]
        if not thinking_steps:
            return 501, [], ""
    except Exception as e:
        log_message(f"Error in parsing obtaining COT: {e}", logger)
        return 501, [], ""

    try:
        json_content = res_text[res_text.find("<json>\n") +
                                7:res_text.find("</json>")]
        if not json_content:
            return 501, [], ""
    except Exception as e:
        log_message(f"Error in parsing LLM response: {e}", logger)
        return 501, [], ""
    return 200, thinking_steps, json_content


def split_json_openAI(res_text: str) -> tuple[int, str]:
    """Split the response text from OpenAI models to extract the molecules
    Note: OpenAI O-series models do not provide Chain of Thoughts (COT) in the response

    Parameters
    ----------
    res_text : str
        The response text from the OpenAI model

    Returns
    -------
    tuple[int, str]
        the status code and json content
    """
    logger = context_logger.get() if ENABLE_LOGGING else None
    try:
        json_content = res_text[res_text.find("<json>\n") +
                                7:res_text.find("</json>")]
        if not json_content:
            return 502, ""

    except Exception as e:
        log_message(f"Error in parsing LLM response: {e}", logger)
        return 502, ""
    return 200, json_content


def split_json_deepseek(res_text: str) -> tuple[int, list[str], str]:
    """Parse the LLM response to extract the thinking steps and json content

    Parameters
    ----------
    res_text : str
        The response text from the LLM model

    Returns
    -------
    tuple[int, list[str], str]
        The status code, thinking steps and json content
    """
    logger = context_logger.get() if ENABLE_LOGGING else None

    try:
        # extract the content within <cot> </cot> tags as thinking content
        thinking_content = res_text[res_text.find("<think>\n") +
                                    6:res_text.find("</think>")]
        if not thinking_content:
            return 503, [], ""

        json_content = res_text[res_text.find("<json>\n") +
                                7:res_text.find("</json>")]
        if not json_content:
            return 503, [], ""

    except Exception as e:
        log_message(f"Error in parsing LLM response: {e}", logger)
        return 503, [], ""
    return 200, [thinking_content], json_content


def split_json_master(res_text: str, model: str) -> tuple[int, list[str], str]:
    """Split the response text based on the model

    Parameters
    ----------
    res_text : str
        The response text from the LLM model
    model : str
        The LLM model used

    Returns
    -------
    tuple[int, list[str], str]
        The status code, thinking steps and json content
    """
    try:
        if model in DEEPSEEK_MODELS:
            status_code, thinking_steps, json_content = split_json_deepseek(
                res_text)
        elif model in OPENAI_MODELS:
            status_code, json_content = split_json_openAI(res_text)
            thinking_steps = []
        else:
            status_code, json_content = split_json_content(
                res_text)
            thinking_steps = []

            # print("-------json_things------------")
            # print(json_content)

    except Exception as e:
        return 505, [], ""

    return status_code, thinking_steps, json_content


def validate_split_json(
        json_content: str) -> tuple[int, list[str], list[str], list[int]]:
    """Validate the split json content from LLM response

    Parameters
    ----------
    json_content : str
        The json content from the LLM response

    Returns
    -------
    tuple[int, list[str], list[str], list[int]]
        The status code, list of molecules, list of explanations and list of confidence scores
    """
    logger = context_logger.get() if ENABLE_LOGGING else None
    try:
        result_list = ast.literal_eval(json_content)
        res_molecules = result_list['data']
        res_explanations = result_list['explanation']
        res_confidence = result_list['confidence_scores']
    except Exception as e:
        log_message(f"Error in parsing response: {e}", logger)
        return 504, [], [], []

    finally:
      print(f"molecules generated => {res_molecules}")
      print(f"explaination => {res_explanations}")
      print(f"confidence {res_confidence}")

    return 200, res_molecules, res_explanations, res_confidence


def llm_pipeline(
    molecule: str,
    LLM: str = "claude-opus-4-20250514",
    messages: Optional[list[dict]] = None,
    stability_flag: str = "False",
    hallucination_check: str = "False",
    use_protecting_group_feature: bool = False,
    local : bool = False
) -> tuple[list[list[str]], list[str], list[float]]:
    """Pipeline to call LLM and validate the results

    Parameters
    ----------
    molecule : str
        The target molecule for retrosynthesis
    LLM : str, optional
        LLM to be used for retrosynthesis , by default "claude-opus-4-20250514"
    messages : Optional[list[dict]], optional
        Conversation history, by default None

    Returns
    -------
    tuple[list[list[str]], list[str], list[float]]
        The output pathways, explanations and confidence scores
    """
    logger = context_logger.get() if ENABLE_LOGGING else None
    output_pathways: list[list[str]] = []
    output_explanations: list[str] = []
    output_confidence: list[float] = []
    run = 0.0
    if stability_flag.lower() == "true" or hallucination_check.lower(
    ) == "true":
        max_run = 1.5
    else:
        max_run = 0.6
    while (output_pathways == [] and run < max_run):
        log_message(f"Calling LLM with molecule: {molecule} and run: {run}",
                    logger)

        # Selecting the model based on the run number
        current_model = LLM
        if LLM in DEEPSEEK_MODELS and run > 0.0:
            current_model = "claude-opus-4-20250514"

        # --------------------
        # Call LLM
        status_code, res_text = call_LLM(
            molecule,
            current_model,
            messages=messages,
            temperature=run,
            use_protecting_group_feature=use_protecting_group_feature,
            local=local)
        if status_code != 200:
            log_message(f"Error in calling LLM: {res_text}", logger)
            run += 1
            get_error_log(status_code)
            continue

        # --------------------
        # Split the response text
        status_code, thinking_steps, json_content = split_json_master(
            res_text, current_model)
        if status_code != 200:
            log_message(f"Error in splitting cot json: {res_text}", logger)
            run += 1
            get_error_log(status_code)
            continue

        # --------------------
        # Validate the split json content
        status_code, res_molecules, res_explanations, res_confidence = validate_split_json(
            json_content)
        if status_code != 200:
            log_message(f"Error in validating split json content: {res_text}",
                        logger)
            run += 1
            get_error_log(status_code)
            continue

        # --------------------
        # Check the validity of the molecules obtained from LLM
        output_pathways, output_explanations, output_confidence = validity_check(
            molecule, res_molecules, res_explanations, res_confidence)

        # --------------------
        # Stability check
        if stability_flag.lower() == "true":
            status_code, stable_pathways = stability_checker(output_pathways)
            if status_code != 200:
                log_message(f"Error in stability check: {stable_pathways}",
                            logger)
                run += 0.1
                get_error_log(status_code)
                continue
            output_pathways = stable_pathways

        # --------------------
        # Hallucination check
        if hallucination_check.lower() == "true":
            log_message(
                f"Calling hallucination check with pathways: {output_pathways}",
                logger)
            status_code, hallucination_pathways = hallucination_checker(
                molecule, output_pathways)
            if status_code != 200:
                log_message(
                    f"Error in hallucination check: {hallucination_pathways}",
                    logger)
                run += 0.1
                get_error_log(status_code)
                continue
            output_pathways = hallucination_pathways

        # --------------------
        # Update the messages for the next call
        # if output_pathways == []:
        #     messages = [{
        #         "role": "system",
        #         "content": SYS_PROMPT
        #     }, {
        #         "role": "user",
        #         "content": USER_PROMPT
        #     }, {
        #         "role": "assistant",
        #         "content": res_text
        #     }, {
        #         "role": "user",
        #         "content": "<add something here>"
        #     }]

        log_message(
            f"Output Pathways: {output_pathways},\n\
                Output Explanations: {output_explanations},\n\
                    Output Confidence: {output_confidence}", logger)
        run += 0.1

    return output_pathways, output_explanations, output_confidence


def get_error_log(status_code: int):
    """Prints error message based on the status code.

    Parameters
    ----------
    status_code : int
        Status Code of the error.
    """
    logger = context_logger.get() if ENABLE_LOGGING else None

    if status_code in ERROR_MAP:
        description = ERROR_MAP[status_code]
        log_message(f"Error Code: {status_code},\n Description: {description}",
                    logger)
    else:
        log_message(f"Error Code: {status_code} is not recognized.", logger)

Overwriting src/utils/llm.py


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 4096


In [ ]:
! wget https://raw.githubusercontent.com/Sauravroy34/DeepRetro/refs/heads/main/data/USPTO_190.csv

--2026-09-28 07:06:18--  https://raw.githubusercontent.com/Sauravroy34/DeepRetro/refs/heads/main/data/USPTO_190.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11859 (12K) [text/plain]
Saving to: ‘USPTO_190.csv’

USPTO_190.csv       100%[===================>]  11.58K  --.-KB/s    in 0s      

2026-09-28 07:06:19 (103 MB/s) - ‘USPTO_190.csv’ saved [11859/11859]



In [ ]:
! ls

aizynthfinder		  reaction_prediction
config			  readme.md
data			  results
deepretro		  retro.code-workspace
docker-compose.yml	  scripts
Dockerfile		  src
docs			  start_backend.sh
env.example		  test_hallucination_model_trainer.ipynb
environment.yml		  test_langfuse_config.py
logs			  tests
model_out		  update_nb.py
molecules.txt		  USPTO_190.csv
MultistepBenchmark.ipynb  uv.lock
notebooks		  viewer
pyproject.toml


In [ ]:
! pip install litellm rootutils diskcache


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 7.7 MB/s eta 0:00:00


In [ ]:
%%writefile /content/DeepRetro/test.py
# load up USPTO-50k test dataset
import os
import pandas as pd
import json
import time
import asyncio
import functools
from concurrent.futures import ThreadPoolExecutor

root_dir = os.getcwd()
from src.main import main
from src.cache import clear_cache_for_molecule

df = pd.read_csv("USPTO_190.csv")
mols_dfs = df['smiles'].to_list()

folder = "USPTO"
for run_no in range(9,12):
  if not os.path.exists(f"{root_dir}/results/dfs/{folder}"):
      os.makedirs(f"{root_dir}/results/dfs/{folder}")
  if not os.path.exists(f"{root_dir}/results/dfs/{folder}/run_{run_no}"):
      os.makedirs(f"{root_dir}/results/dfs/{folder}/run_{run_no}")

  for mol in mols_dfs:
      molecule = mol
      print(f"Running {mol}")

      llm = "Qwen/Qwen2.5-0.5B-Instruct"
      az_model = "USPTO"

      time1 = time.time()
      try:
          clear_cache_for_molecule(molecule)
          print(f"Running {molecule} with {llm} and {az_model}")

          res_dict = main(molecule ,llm=llm, az_model=az_model, stability_flag="True", hallucination_check="True", local = True)

          with open(f"{root_dir}/results/dfs/{folder}/run_{run_no}/{mol}_stability_hallucination.json", "w") as f:
              json.dump(res_dict, f, indent=4)
      except Exception as e:
          print("Error in molecule:", mol)
          print("Error:", e)
      time2 = time.time()
      print(f"Time taken for {mol} on {folder}: {time2-time1} seconds")

Overwriting /content/DeepRetro/test.py


In [ ]:
!python /content/DeepRetro/test.py

Streaming output truncated to the last 5000 lines.
retro reaction from template [C:2]-[CH2;D2;+0:1]-[N;H0;D2;+0:3]=[#7;+:4]=[N;-;D1;H0:5]>>C-S(=O)(=O)-O-[CH2;D2;+0:1]-[C:2].[N-;H0;D1:3]=[#7;+:4]=[N;-;D1;H0:5] on molecule [N-]=[N+]=NC[C@H]1CC[C@@H](C(=O)Nc2cccc(OC(F)(F)F)c2)N1C(=O)OCc1ccccc1 was filtered out with prob 0.0006826817989349365
retro reaction from template [C;D1;H3:2]-[C;H0;D3;+0:1](=[O;D1;H0:3])-[NH;D2;+0:4]-[c:5]>>O-[C;H0;D3;+0:1](-[C;D1;H3:2])=[O;D1;H0:3].[NH2;D1;+0:4]-[c:5] on molecule [N-]=[N+]=NC[C@H]1CC[C@@H](C(=O)Nc2cccc(OC(F)(F)F)c2)N1C(=O)OCc1ccccc1 did not produce any reactants
retro reaction from template [C:2]-[CH2;D2;+0:1]-[N;H0;D2;+0:3]=[#7;+:4]=[N;-;D1;H0:5]>>O-[CH2;D2;+0:1]-[C:2].O=P(-[N;H0;D2;+0:3]=[#7;+:4]=[N;-;D1;H0:5])(-c1:c:c:c:c:c:1)-c1:c:c:c:c:c:1 on molecule [N-]=[N+]=NC[C@H]1CC[C@@H](C(=O)Nc2cccc(OC(F)(F)F)c2)N1C(=O)OCc1ccccc1 was filtered out with prob 0.0017853081226348877
retro reaction from template [#7:3]-[C:2](=[O;D1;H0:4])-[CH2;D2;+0:1]-[N;H0